# День 2 — Очистка текста и подготовка признаков

**Цель:** получить воспроизводимый набор данных для последующего ML pipeline, не удаляя значимые финансовые обозначения.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from finnews_sentiment.data.load_data import load_financial_phrasebank, save_processed
from finnews_sentiment.features.preprocess import (
    LABEL_MAPPING,
    clean_text,
    prepare_news_data,
)

pd.set_option("display.max_colwidth", 120)

## 1. Загрузка результата дня 1

Если локального CSV ещё нет, воспроизводим его тем же загрузчиком, который использовался в первом ноутбуке.

In [2]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / "pyproject.toml").exists() else current_dir.parent
assert (PROJECT_ROOT / "pyproject.toml").exists(), "Запустите ноутбук из корня проекта или каталога notebooks"

RAW_PATH = PROJECT_ROOT / "data/raw/financial_phrasebank_75agree.csv"
PROCESSED_PATH = PROJECT_ROOT / "data/processed/clean_news.csv"

if RAW_PATH.exists():
    raw_df = pd.read_csv(RAW_PATH, dtype="string")
else:
    raw_df = load_financial_phrasebank()
    save_processed(raw_df, RAW_PATH)

print(f"Исходный размер: {raw_df.shape[0]} строк × {raw_df.shape[1]} столбца")
print(f"Пропусков в тексте: {raw_df['text'].isna().sum()}")
print(f"Пустых текстов: {raw_df['text'].fillna('').str.strip().eq('').sum()}")
print(f"Полных дублей: {raw_df.duplicated().sum()}")
display(raw_df.head())

Исходный размер: 3453 строк × 2 столбца
Пропусков в тексте: 0
Пустых текстов: 0
Полных дублей: 5


,text,sentiment
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...",neutral
1,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,positive
2,"For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from EUR76m for the same period a year ear...",positive
3,"In the third quarter of 2010 , net sales increased by 5.2 % to EUR 205.5 mn , and operating profit by 34.9 % to EUR ...",positive
4,Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding period in 2007 representing 7.7 % of net s...,positive


## 2. Ручная проверка очистки

Очистка убирает HTML и лишние пробелы, нормализует регистр, но сохраняет числа, валюты, проценты и пунктуацию.

In [3]:
manual_examples = pd.DataFrame({
    "До": [
        "  <b>PROFIT&nbsp;rose</b> 10% to $5.00  ",
        "Revenue\nincreased\tto EUR 12.5 mn.",
        "Market remains STABLE despite volatility.",
        None,
    ]
})
manual_examples["После"] = manual_examples["До"].map(clean_text)
display(manual_examples)

,До,После
0,<b>PROFIT&nbsp;rose</b> 10% to $5.00,profit rose 10% to $5.00
1,Revenue\nincreased\tto EUR 12.5 mn.,revenue increased to eur 12.5 mn.
2,Market remains STABLE despite volatility.,market remains stable despite volatility.
3,None,


## 3. Единый preprocessing step

Функция проверяет схему и метки, удаляет пустые тексты и дубли, кодирует классы и добавляет базовые признаки.

In [4]:
processed_df = prepare_news_data(raw_df)
duplicates_removed = len(raw_df) - len(processed_df)

print(f"Обработанный размер: {processed_df.shape[0]} строк × {processed_df.shape[1]} столбцов")
print(f"Удалено пустых строк и дублей: {duplicates_removed}")
display(processed_df.head())

Обработанный размер: 3448 строк × 7 столбцов
Удалено пустых строк и дублей: 5


,text,sentiment,text_clean,label,word_count_clean,char_count,dollar_count
0,"According to Gran , the company has no plans to move all production to Russia , although that is where the company i...",neutral,"according to gran , the company has no plans to move all production to russia , although that is where the company i...",1,25,127,0
1,With the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,positive,with the new production plant the company would increase its capacity to meet the expected increase in demand and wo...,2,33,206,0
2,"For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from EUR76m for the same period a year ear...",positive,"for the last quarter of 2010 , componenta 's net sales doubled to eur131m from eur76m for the same period a year ear...",2,39,193,0
3,"In the third quarter of 2010 , net sales increased by 5.2 % to EUR 205.5 mn , and operating profit by 34.9 % to EUR ...",positive,"in the third quarter of 2010 , net sales increased by 5.2 % to eur 205.5 mn , and operating profit by 34.9 % to eur ...",2,29,125,0
4,Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding period in 2007 representing 7.7 % of net s...,positive,operating profit rose to eur 13.1 mn from eur 8.7 mn in the corresponding period in 2007 representing 7.7 % of net s...,2,24,122,0


## 4. Проверка результата

In [5]:
assert len(processed_df) == 3448
assert duplicates_removed == 5
assert not processed_df['text_clean'].eq('').any()
assert not processed_df['text_clean'].isna().any()
assert not processed_df.duplicated(['text_clean', 'sentiment']).any()
assert processed_df['label'].notna().all()
assert set(processed_df['sentiment']) == set(LABEL_MAPPING)

quality_summary = pd.DataFrame({
    "Количество": processed_df['sentiment'].value_counts(),
    "Числовая метка": pd.Series(LABEL_MAPPING),
}).sort_values('Числовая метка')
display(quality_summary)
display(processed_df[['word_count_clean', 'char_count', 'dollar_count']].describe().round(2))

,Количество,Числовая метка
negative,420,0
neutral,2141,1
positive,887,2


,word_count_clean,char_count,dollar_count
count,3448.00,3448.00,3448.00
mean,22.75,124.80,0.06
std,10.05,56.31,0.32
min,2.00,9.00,0.00
25%,15.00,81.00,0.00
50%,21.00,116.00,0.00
75%,29.00,160.00,0.00
max,81.00,315.00,4.00


## 5. Сохранение обработанных данных

In [6]:
save_processed(processed_df, PROCESSED_PATH)
print(f"Обработанный датасет сохранён: {PROCESSED_PATH.relative_to(PROJECT_ROOT)}")

Обработанный датасет сохранён: data/processed/clean_news.csv


## 6. Выводы

- Получен единый переиспользуемый preprocessing step.
- После удаления пяти дублей осталось **3448** наблюдений.
- Метки представлены и строковым классом, и фиксированным числом.
- Числа, валюты и проценты сохранены для будущего TF-IDF baseline.
- Лемматизация и удаление стоп-слов не включены в baseline и могут быть отдельно проверены как улучшение.